# 🌧️ IndabaX Guinée 2026 - Prédiction des Précipitations à 24h
Ce notebook présente la démarche complète de science des données pour le hackathon :
1. **Exploration et Diagnostic des Données** (nettoyage des 20 villes, déséquilibre de classes, gestion temporelle).
2. **Ingénierie des Variables Météorologiques & Climatologiques** (Point de rosée, déficit de pression de vapeur, projections Harmattan / Mousson, dynamique du FIT).
3. **Validation Croisée Robuste** (Stratified K-Fold & validation temporelle).
4. **Modélisation Avancée LightGBM** et calibration multi-classe.
5. **Évaluation & Génération de la Soumission**.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import log_loss, classification_report

from src.config import TRAIN_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH, CLASSES, TARGET_MAP
from src.data_cleaning import clean_dataframe
from src.stations_metadata import get_stations_df
from src.feature_engineering import extract_features
from src.evaluate import evaluate_against_reference

print("Modules importés avec succès !")

## 1. Chargement et Aperçu des Stations Guinéennes

In [ ]:
stations_df = get_stations_df()
print(f"Nombre de stations météo guinéennes intégrées : {len(stations_df)}")
stations_df.head(10)

## 2. Chargement et Nettoyage des Relevés Météo

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)

train = clean_dataframe(train_raw)
test = clean_dataframe(test_raw)

print(f"Train : {train.shape[0]} lignes de {train['timestamp'].min()} à {train['timestamp'].max()}")
print(f"Test  : {test.shape[0]} lignes de {test['timestamp'].min()} à {test['timestamp'].max()}")

print("\nRépartition de la cible (target) :")
display(train['target'].value_counts(normalize=True).round(4) * 100)

## 3. Extraction des Variables Métier et Temporelles

In [ ]:
train['is_test'] = 0
test['is_test'] = 1
combined = pd.concat([train, test], ignore_index=True)

combined_feat, features = extract_features(combined, categorical_as_category=True)
print(f"Nombre de variables explicatives générées : {len(features)}")
print(features)

## 4. Évaluation du Modèle et Génération de Soumission
Exécutez `scripts/train_advanced.py` pour lancer l'entraînement complet multi-fold.